# 🎮 SheepBell - Google Colab 起動ノートブック
ゲームLQA用 自動動画クリップ & 文字起こしツール (Gradio UI)

### 📌 全体の流れ
1. **ランタイムの設定**: メニューの **「ランタイム」 > 「ランタイムのタイプを変更」** で **T4 GPU** が選択されていることを確認します。
2. **セル 1〜3 を順番に実行**: 発行された `https://xxxx.gradio.live` のリンクを開いて動画を処理します。
3. **セル 4 を実行 (後処理)**: Google Drive に日付フォルダを自動作成し、生成物（動画クリップ・画像・CSV）の保存とスプレッドシート作成を一発で行います。

### 1. (推奨) Google Drive のマウント
成果物（クリップ動画・画像・スプレッドシート）を Google Drive に保存するため、最初に実行してアクセスを許可してください。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. リポジトリのクローン / 最新化 (初回はclone、2回目以降はpull)

In [ ]:
import os

# 自身のGitHubリポジトリURLに変更してください
REPO_URL = "https://github.com/your-username/SheepBell.git"
TARGET_DIR = "/content/SheepBell"

if not os.path.exists(TARGET_DIR):
    print("🚀 リポジトリをクローン中...")
    !git clone {REPO_URL} {TARGET_DIR}
else:
    print("🔄 既存リポジトリを最新に更新 (git pull) 中...")
    %cd {TARGET_DIR}
    !git pull

%cd {TARGET_DIR}

# 依存パッケージのインストール (GPUライブラリ含む)
!pip install -q gradio faster-whisper silero-vad onnxruntime-gpu imageio-ffmpeg gspread nvidia-cublas-cu12 nvidia-cudnn-cu12

### 3. Web UI の起動 (Gradio Share URL 発行)
実行すると一番下に `Running on public URL: https://xxxx.gradio.live` と表示されるので、そのリンクをクリックしてブラウザで利用します。
*(※UIの利用中は、このセルを実行中のままにしておきます)*

In [ ]:
# GPU上でGradio UIを起動し、公開URLを発行します
!python app.py --share

---
## 🎁 処理完了後の後処理ツール
Web UIでの処理が終わったら、下のセルを実行してください。

### 4. 🚀 全成果物（動画・画像・CSV）を Google Drive に保存 ＆ スプレッドシートを自動作成
Colabの一時領域にある生成物を Google Drive の指定フォルダに丸ごとコピーし、同時に Google スプレッドシートを作成してリンクを発行します。

In [ ]:
import os
import csv
import shutil
import datetime
from pathlib import Path
from google.colab import auth
from google.auth import default
import gspread

# ==========================================
# 設定: Google Drive 内の保存先フォルダ名
# ==========================================
DRIVE_BASE_DIR = "/content/drive/MyDrive/LQA_Results"
SRC_ISSUES_DIR = "/content/SheepBell/issues"

# 1. 日時フォルダの作成 (例: /content/drive/MyDrive/LQA_Results/20261006_1105)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
target_drive_folder = Path(DRIVE_BASE_DIR) / f"LQA_{timestamp}"
target_drive_folder.mkdir(parents=True, exist_ok=True)

# 2. 生成された全ファイル (動画クリップ, 画像, CSV, JSON) を Google Drive にコピー
if os.path.exists(SRC_ISSUES_DIR):
    for item in os.listdir(SRC_ISSUES_DIR):
        s = os.path.join(SRC_ISSUES_DIR, item)
        d = os.path.join(str(target_drive_folder), item)
        if os.path.isfile(s):
            shutil.copy2(s, d)
    print(f"📁 全ファイルを Google Drive に保存しました: {target_drive_folder}")
else:
    print("⚠️ issues フォルダが見つかりません。")

# 3. Google アカウント認証 & スプレッドシート作成
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

csv_path = target_drive_folder / "lqa_issues.csv"
if not csv_path.exists():
    csv_path = Path(SRC_ISSUES_DIR) / "lqa_issues.csv"

if csv_path.exists():
    with open(csv_path, "r", encoding="utf-8-sig") as f:
        rows = list(csv.reader(f))

    sheet_title = f"LQA_Issues_{timestamp}"
    sh = gc.create(sheet_title)
    worksheet = sh.get_worksheet(0)
    worksheet.update(rows)

    print("\n🎉 Google スプレッドシートを作成しました！")
    print(f"🔗 スプレッドシートを開く -> {sh.url}")
else:
    print("⚠️ CSV ファイルが見つかりませんでした。")

### 5. 📦 (任意) PCローカルへ Zip 一括ダウンロード
Google Drive を使わず、手元のPCに直接 Zip でダウンロードしたい場合は実行してください。

In [ ]:
!zip -r /content/issues.zip /content/SheepBell/issues
from google.colab import files
files.download("/content/issues.zip")